In [11]:
# 標準函式庫
import sys
import os
import warnings
import cv2


# 第三方函式庫 (科學計算/優化)
import emcee
import numpy as np
import scipy.constants as spc
from scipy.interpolate import interp1d
from scipy.optimize import fsolve # 假設你需要 fsolve

# 天文學/數據處理函式庫
from astropy import units as u
from astropy.io import fits 
from astropy.wcs import WCS
from spectral_cube import SpectralCube

# 繪圖函式庫
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import corner
from matplotlib.colors import PowerNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# 專案本地模組
import PSSpy as pss

In [12]:
ra_start = '19:01:06.961'
ra_end = '19:01:10.248'
dec_deg = 36 + 57/60 + 20/3600  # +31.21.38.180 => 31.3606 度
distance_pc = 160
n_pixels = 789

arcsec_range, AU_per_pixel = pss.calc_ra_arcsec(ra_start, ra_end, dec_deg, distance_pc, n_pixels)

print(f"RA arcsec range (rounded): {arcsec_range} arcsec")
print(f"AU per pixel: {AU_per_pixel:.2f} AU/pixel")

RA arcsec range (rounded): 39 arcsec
AU per pixel: 7.91 AU/pixel


In [ ]:

cube = SpectralCube.read('S_CrA_13CO_spw25_tav_jupyter_shifted.fits')
header = fits.getheader('Per-emb-50_CD_l021l060_uvsub_H2CO_multi_small_fitcube.fits')

im_center = int(header['CRPIX2']), int(header['CRPIX1'])  # (z, x)
v0 = header['CRVAL3']          # reference velocity
dx = abs(header['CDELT1'])      # arcsec per pixel
dz = abs(header['CDELT2'])      # arcsec per pixel
dv = abs(header['CDELT3'])      # km/s per channel
# v_lastch_vel = 4.0984
v_lastch_num = 150

# Subcube and moment calculation
velocity_range = [2.4926, 14.8636] * u.km / u.s  # Adjusted velocity range
subcube = cube.spectral_slab(velocity_range[0], velocity_range[1])
moment0 = subcube.moment(order=0).value
moment1 = subcube.moment(order=1).value
rms_channel = 0.026211100061251217
rms_mom0 = 5.134089023394e-2
# mom0_3rms = np.ma.masked_where(moment0 < 3.5*rms_channel, moment0)
# mom1_3rms = np.ma.masked_where(moment0 < 3.5*rms_channel, moment1)

/opt/anaconda3/lib/python3.11/site-packages/spectral_cube/_moments.py:168: RuntimeWarning: invalid value encountered in divide
  return (np.nansum(data * pix_cen, axis=axis) /


In [14]:
hdu_mom0 = fits.PrimaryHDU(data=moment0, header=header)
hdu_mom0.writeto('S_CrA_13CO_mom0.fits', overwrite=True)
hdu_mom1 = fits.PrimaryHDU(data=moment1.data, header=header)
hdu_mom1.header['BUNIT'] = 'km/s'
hdu_mom1.writeto('S_CrA_13CO_mom1.fits', overwrite=True)

In [15]:
find_streamcom = np.array([[396, 396], [371, 355], [340, 355], [309, 369], [279, 389], [257, 463]]) - (396, 396)

In [16]:
find_r, find_theta = pss.spherical_coords(find_streamcom[:, 0], find_streamcom[:, 1])
find_streaml = interp1d(find_r, find_theta)

In [17]:
center1 = (390, 395)  # (y, x)
center2 = (368, 383)  # (y, x)

new_center = int((center1[0] + center2[0]) / 2), int((center1[1] + center2[1]) / 2) # (y, x)
ori_center = (396, 396)

ny, nx = moment0.shape
radius = 40

# 建立 2D mask 並擴展成 3D mask
mask2d = pss.circular_mask((ny, nx), new_center, radius)
mask3d = np.repeat(mask2d[np.newaxis, :, :], cube.shape[0], axis=0)

# 套用到 cube
masked_center_cube = cube.with_mask(mask3d)
# masked_center_cube = masked_center_cube.with_fill_value(0)

In [18]:
# cube_data = cube.filled_data[:].value  # shape = (z, y, x)
maskcent_cube_data = masked_center_cube.filled_data[:].value  # shape = (z, y, x)

init_points = [(35, new_center[0], new_center[1]), (35, 355, 371), (35, 355, 340), (35, 369, 309), (35, 389, 279), (35, 463, 257)]  # (z, y, x) coordinates of initial points
# stream_mask = pss.grow_region(cube_data, init_points, rms_channel, sigma_thresh=3.5, max_iter=1000)
maskcent_stream_mask = pss.grow_region(maskcent_cube_data, init_points, rms_channel, sigma_thresh=3.5, max_iter=1000)

In [19]:
# masked_cube_ori = cube.with_mask(stream_mask)
masked_cube = masked_center_cube.with_mask(maskcent_stream_mask)
# masked_cube = masked_cube.with_fill_value(0)

str_moment0 = masked_cube.moment(order=0).value
str_moment1 = masked_cube.moment(order=1).value
# str_moment0_ori = masked_cube_ori.moment(order=0).value
# str_moment1_ori = masked_cube_ori.moment(order=1).value

/opt/anaconda3/lib/python3.11/site-packages/spectral_cube/_moments.py:168: RuntimeWarning: invalid value encountered in divide
  return (np.nansum(data * pix_cen, axis=axis) /


In [20]:
ty, tx = (ori_center[0] - new_center[0], ori_center[1] - new_center[1])

# Create the 2x3 translation matrix
# M = [[1, 0, tx],
#      [0, 1, ty]]
M = np.float32([[1, 0, tx], [0, 1, ty]])

# Get the image dimensions
rows, cols= str_moment0.shape

# Apply the affine transformation
shifted_str_mom0 = np.ma.masked_invalid(cv2.warpAffine(str_moment0, M, (cols, rows), borderValue=np.nan))
shifted_str_mom1 = np.ma.masked_invalid(cv2.warpAffine(str_moment1, M, (cols, rows), borderValue=np.nan))
# shifted_str_mom0_ori = np.ma.masked_invalid(cv2.warpAffine(str_moment0_ori, M, (cols, rows), borderValue=np.nan))
# shifted_str_mom1_ori = np.ma.masked_invalid(cv2.warpAffine(str_moment1_ori, M, (cols, rows), borderValue=np.nan))

In [ ]:
hdu_mom0 = fits.PrimaryHDU(data=str_moment0, header=header)
hdu_mom0.writeto('S_CrA_13CO_streamer_mom0.fits', overwrite=True)

In [ ]:
ny, nx = shifted_str_mom0.shape # 或者 im_cube.shape[0], im_cube.shape[1]
delx_arcsec = dx * 3600.
delz_arcsec = dz * 3600.

x_left_arcsec = (0 - new_center[1]) * delx_arcsec  # 0 像素 - 中心像素 = 相對左邊界像素偏移
x_right_arcsec = (nx - 1 - new_center[1]) * delx_arcsec # 最右像素 - 中心像素 = 相對右邊界像素偏移
y_bottom_arcsec = (0 - new_center[0]) * delz_arcsec # 最下像素 - 中心像素 = 相對下邊界像素偏移
y_top_arcsec = (ny - 1 - new_center[0]) * delz_arcsec # 最上像素 - 中心像素 = 相對上邊界像素偏移
extent = (x_left_arcsec - 0.5 * delx_arcsec, x_right_arcsec + 0.5 * delx_arcsec,
          y_bottom_arcsec - 0.5 * delz_arcsec, y_top_arcsec + 0.5 * delz_arcsec) # image extent

# plot, relative coordinate
# figure
fig = plt.figure(figsize=(8.27, 8.27))
ax  = fig.add_subplot(111)
# color
norm = PowerNorm(gamma=0.5, vmin=0, vmax=1.8)  # Adjust gamma for better visibility
imcolor = ax.imshow(shifted_str_mom0, origin='lower', cmap='inferno', extent=extent, norm=norm)
# color bar
divider = make_axes_locatable(ax)
cax     = divider.append_axes('right', size='3%', pad='1%')
cbar = fig.colorbar(imcolor, cax=cax)
cbar.set_label('(Jy/beam km/s)',fontsize=20)
ax.scatter(0, 0, c='w', s=100, marker='+')
# range
ax.set_xlim(12.5, -12.5)
ax.set_ylim(-12.5, 12.5)

# axis label
ax.set_title('S CrA streamer'+r'$\rm ^{13}CO$'+' origin',fontsize=30)
ax.set_xlabel('RA Offset (arcsec)',fontsize=20)
ax.set_ylabel('DEC Offset (arcsec)',fontsize=20)
plt.show()